# Comprehensive Technical Documentation: End-to-End Local RAG System


## Environment Initialization & Dependency Setup

Before building the pipeline, the runtime environment must be provisioned with specialized libraries to support PDF parsing, dense vector generation, fast nearest-neighbor lookups, and local LLM execution.

- transformers & accelerate: Core Hugging Face libraries utilized to download, configure, and optimize local model deployment across hardware constraints (such as GPU memory allocation).

- datasets & evaluate: Frameworks to manage data flows and systematically track modeling metrics.

- pypdf: A lightweight, pure-Python library engineered to extract raw layout text from unstructured PDF documents.

- sentence-transformers: Built on top of PyTorch, this package provides access to pre-trained bi-encoder architectures optimized for generating fixed-size dense embeddings that reflect deep sentence-level semantic meaning.

- faiss-cpu: Facebook AI Similarity Search. A highly optimized C++ library with Python bindings designed for high-performance vector clustering and efficient, low-latency similarity searches on CPU environments.

In [ ]:
!pip install transformers datasets faiss-cpu sentence-transformers pymupdf  langchain langchain-text-splitters


## Document Processing & Text Segmentation Layer



###   PDF Parsing

The system reads local scientific manuscripts—specifically the foundational papers « Attention Is All You Need » and « BERT: Pre-training of Deep Bidirectional Transformers »—and extracts raw character streams page-by-page.

In [ ]:

import urllib.request
import fitz  # pymupdf

papers = [
    {
        "title": "Attention Is All You Need",
        "url"  : "https://arxiv.org/pdf/1706.03762"
    },
    {
        "title": "BERT: Pre-training of Deep Bidirectional Transformers",
        "url"  : "https://arxiv.org/pdf/1810.04805"
    },
    {
        "title": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
        "url"  : "https://arxiv.org/pdf/2005.11401"
    },
]

documents = []

for paper in papers:
    filename = paper["title"].replace(" ", "_")[:30] + ".pdf"

    # download
    urllib.request.urlretrieve(paper["url"], filename)

    # extract text
    doc  = fitz.open(filename)
    text = ""
    for page in doc:
        text += page.get_text()

    documents.append({
        "title": paper["title"],
        "text" : text
    })

    print(f"✅ {paper['title']}")
    print(f"   {len(doc)} pages — {len(text.split())} words\n")

print(f"Knowledge base ready: {len(documents)} papers")
print(f"Total words: {sum(len(d['text'].split()) for d in documents)}")

✅ Attention Is All You Need
   15 pages — 6095 words

✅ BERT: Pre-training of Deep Bidirectional Transformers
   16 pages — 10152 words

✅ Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks
   19 pages — 9886 words

Knowledge base ready: 3 papers
Total words: 26133


### Dynamic Sliding-Window Chunking Strategy
Raw text strings cannot be passed directly into an embedding model due to strict context length tokens constraints (typically 512 tokens for standard transformer encoders). Furthermore, chunking is required to isolate localized context.

We implement a custom character-based sliding-window text splitter:

Chunk Size: Splitting text into chunks based on character thresholds.

Chunk Overlap: A portion of text is repeated between consecutive chunks to prevent information loss across sentence boundaries.

Output: This process transforms our massive text documents into 361 distinct, continuous text segments (chunks).

In [ ]:

from langchain_text_splitters import RecursiveCharacterTextSplitter

# this splitter tries to cut at natural breaks (paragraphs, sentences)
# instead of cutting randomly in the middle of a sentence
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # ~500 characters per chunk
    chunk_overlap=150,      # 150 characters overlap between chunks
    separators=["\n\n", "\n", ". ", " ", ""]  # tries these in order
)

In [ ]:
def clean_text(text):
    # fix common PDF ligatures
    ligature_map = {
        'ﬁ': 'fi', 'ﬂ': 'fl', 'ﬀ': 'ff',
        'ﬃ': 'ffi', 'ﬄ': 'ffl',
        '–': '-', '—': '-'
    }
    for bad, good in ligature_map.items():
        text = text.replace(bad, good)

    # remove hyphen-linebreak artifacts like "fine- tuned" → "fine-tuned"
    text = re.sub(r'-\s+', '-', text)

    # remove email addresses
    text = re.sub(r'\S+@\S+', '', text)

    # remove short noisy lines (headers/names)
    lines = text.split('\n')
    cleaned_lines = []
    for line in lines:
        line = line.strip()
        if len(line) < 40 and not any(p in line for p in ['.', ',', ';']):
            continue
        if line:
            cleaned_lines.append(line)
    text = ' '.join(cleaned_lines)

    # collapse multiple spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


In [ ]:
import re  # ◄ Add this line!
# re-clean and re-chunk
for doc in documents:
    doc["text"] = clean_text(doc["text"])


all_chunks = []
for doc in documents:
    chunks = splitter.split_text(doc["text"])
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "text"    : chunk,
            "source"  : doc["title"],
            "chunk_id": i
        })

print(f"✅ Total chunks: {len(all_chunks)}")

✅ Total chunks: 458


In [ ]:
# check a few random chunks to confirm they look clean
import random

sample_chunks = random.sample(all_chunks, 5)
for c in sample_chunks:
    print(f"── {c['source']} (chunk {c['chunk_id']}) ──")
    print(c['text'][:200])
    print()

── Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks (chunk 126) ──
. Association for Computational Linguistics. doi: 10.18653/v1/P19-1612. URL https://www.aclweb.org/ anthology/P19-1612. [32] Mike Lewis, Yinhan Liu, Naman Goyal, Marjan Ghazvininejad, Abdelrahman Moha

── Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks (chunk 66) ──
. The posterior for document 1 is high when generating “A Farewell to Arms" and for document 2 when generating “The Sun Also Rises". Table 3: Examples from generation tasks. RAG models generate more s

── BERT: Pre-training of Deep Bidirectional Transformers (chunk 64) ──
. 10https://gluebenchmark.com/leaderboard Wikipedia containing the answer, the task is to predict the answer text span in the passage. As shown in Figure 1, in the question answer-ing task, we represe

── BERT: Pre-training of Deep Bidirectional Transformers (chunk 28) ──
. The question-answering example in Figure 1 will serve as a running example for this s

##  Vectorization & Semantic Indexing (The Vector Database)

### Embedding Model Configuration

To bridge the gap between human language and mathematical space, we instantiate the sentence-transformers/all-MiniLM-L6-v2 bi-encoder model.

Dimensionality: This model maps every textual chunk into a dense numerical vector of 384 continuous features.

Semantic Preservation: It is fine-tuned on an immense web corpus to ensure that sentences with close conceptual contexts (e.g., "unsupervised tasks" and "pre-trained on text") reside in close proximity within the vector space.

In [ ]:
from sentence_transformers import SentenceTransformer

# small, fast, well-performing embedding model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

texts = [chunk["text"] for chunk in all_chunks]

embeddings = embedder.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(f"✅ Embeddings created")
print(f"   Shape: {embeddings.shape}")  # (num_chunks, 382)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

✅ Embeddings created
   Shape: (458, 384)


In [ ]:
import faiss
import numpy as np

# make sure embeddings are float32 (FAISS requirement)
embeddings = np.array(embeddings).astype('float32')

# get the dimension of your vectors (384 for all-MiniLM-L6-v2)
dimension = embeddings.shape[1]

# create a simple FAISS index using L2 (Euclidean) distance
index = faiss.IndexFlatL2(dimension)

# add all your chunk embeddings to the index
index.add(embeddings)

print(f"✅ FAISS index created")
print(f"   Number of vectors indexed: {index.ntotal}")
print(f"   Vector dimension: {dimension}")

✅ FAISS index created
   Number of vectors indexed: 458
   Vector dimension: 384


### Indexing with FAISS (IndexFlatL2)

Once all 458 text chunks are transformed into 458 * 384 matrices, they are indexed into a local FAISS database.Index Architecture:

 We employ faiss.IndexFlatL2, which performs a brute-force, exact search.

In [ ]:
def search(query, top_k=5):
    # convert the query into a vector using the SAME embedding model
    query_vector = embedder.encode([query]).astype('float32')

    # search the index for the top_k closest vectors
    distances, indices = index.search(query_vector, top_k)

    # retrieve the actual text chunks
    results = []
    for idx, dist in zip(indices[0], distances[0]):
        results.append({
            "text"    : all_chunks[idx]["text"],
            "source"  : all_chunks[idx]["source"],
            "distance": float(dist)
        })
    return results

# try it!
results = search("How many attention heads does the Transformer use?")

for i, r in enumerate(results):
    print(f"── Result {i+1} (distance: {r['distance']:.2f}) ──")
    print(f"Source: {r['source']}")
    print(f"Text  : {r['text']}\n")

── Result 1 (distance: 0.56) ──
Source: Attention Is All You Need
Text  : . This makes it more difficult to learn dependencies between distant positions [12]. In the Transformer this is reduced to a constant number of operations, albeit at the cost of reduced effective resolution due to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as described in section 3.2

── Result 2 (distance: 0.66) ──
Source: Attention Is All You Need
Text  : . In this work we employ h = 8 parallel attention layers, or heads. For each of these we use dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost is similar to that of single-head attention with full dimensionality. 3.2.3 The Transformer uses multi-head attention in three different ways: • In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values come from the output of the encoder

── Result 3 (distance: 0.87

During development, we observed that with chunk_overlap=50, specific numerical facts (such as the number of attention heads, h=8) were sometimes split across chunk boundaries, separating the value from its explanation. Increasing chunk_overlap to 150 resolved this issue, ensuring that key facts remained intact within a single chunk. This highlights the importance of tuning the chunking strategy based on the structure of the source documents, particularly for technical papers with dense mathematical notation.

## Local Generative Model Configuration (Llama-3)

To enable reasoning without reliance on external commercial cloud APIs, we host a local, quantized instance of Meta's Llama-3.Quantization:

 The model operates under low-precision constraints to significantly reduce the VRAM/RAM footprint while retaining high linguistic fluency.

 Hyperparameter Tuning:

 - Max New Tokens: Limited to a strict generation window to ensure succinct answers.
 - Temperature (T \to 0): Maintained close to 0 to minimize creativity and restrict the model to deterministic, literal context interpretation.

In [ ]:
def build_prompt(query, retrieved_chunks):
    context = "\n\n".join([f"[{c['source']}]: {c['text']}" for c in retrieved_chunks])

    prompt = f"""Answer the question based only on the context below.
If the answer is not in the context, say "I don't know based on the provided documents."

Context:
{context}

Question: {query}

Answer:"""
    return prompt

# test it
results = search("How many attention heads does the Transformer use?", top_k=3)
prompt = build_prompt("How many attention heads does the Transformer use?", results)

print(prompt)

Answer the question based only on the context below. 
If the answer is not in the context, say "I don't know based on the provided documents."

Context:
[Attention Is All You Need]: . This makes it more difficult to learn dependencies between distant positions [12]. In the Transformer this is reduced to a constant number of operations, albeit at the cost of reduced effective resolution due to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as described in section 3.2

[Attention Is All You Need]: . In this work we employ h = 8 parallel attention layers, or heads. For each of these we use dk = dv = dmodel/h = 64. Due to the reduced dimension of each head, the total computational cost is similar to that of single-head attention with full dimensionality. 3.2.3 The Transformer uses multi-head attention in three different ways: • In "encoder-decoder attention" layers, the queries come from the previous decoder layer, and the memory keys and values c

In [ ]:
# !pip uninstall -y transformers
# !pip install transformers

In [ ]:
import transformers
print(transformers.__version__)

5.12.1


In [ ]:
def rag_answer(query, top_k=3, max_new_tokens=150):
    retrieved = search(query, top_k=top_k)
    context = "\n".join([c["text"] for c in retrieved])

    prompt = f"""Answer the question based only on the context below.

Context: {context}

Question: {query}

Answer:"""

    answer = generate_answer(prompt, max_new_tokens=max_new_tokens)

    return {
        "query"  : query,
        "answer" : answer,
        "sources": [c["source"] for c in retrieved]
    }

def answer_without_rag(query, max_new_tokens=150):
    return generate_answer(query, max_new_tokens=max_new_tokens)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

model_name = "google/flan-t5-large"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

print("✅ Local LLM loaded (works on transformers v5)")

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


✅ Local LLM loaded (works on transformers v5)


Title: Base Inference Prompt Engineering

Description: Defines a foundational text generation function (generate_answer) that directly formats and routes clean string prompts to the underlying Llama-3 model.

In [ ]:
def generate_answer(prompt, max_new_tokens=150):
    inputs  = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)
# test with a simple factual question — closer to your real use case
print(generate_answer("What is the capital of France?"))

paris


Title: Baseline Evaluation (Without RAG)

Description: Evaluates the standalone generative performance of the LLM by submitting specific technical questions about Transformer architectures without injecting external documents, exposing its internal knowledge limits and tendency to hallucinate

In [ ]:
query = "How many attention heads does the Transformer use?"

print("🚫 Without RAG:")
print(answer_without_rag(query))

print("\n✅ With RAG:")
result = rag_answer(query)
print(result["answer"])
print(f"(source: {set(result['sources'])})")

🚫 Without RAG:
three

✅ With RAG:
8
(source: {'Attention Is All You Need'})


## The Core RAG Inference Pipeline

The system coordinates a multi-stage retrieval and execution chain whenever a user submits a prompt:

[User Question]

▼

[all-MiniLM-L6-v2] ──► Generates Question Vector (1 x 384)

      
▼

  [FAISS Index]    ──► Calculates L2 Distance across 361 Chunks
       
▼

 [Top-K Contexts]  ──► Extracts Top k Text Segments

▼

[Prompt Engineer]  ──► Packs Context + Question into Instruction Template

▼

   [Llama-3]       ──► Outputs Factually Grounded Text

In [ ]:
test_questions = [
    "How many attention heads does the Transformer use?",
    "How many encoder layers does the base Transformer have?",
    "What corpus was BERT pre-trained on?",
    "What is retrieval augmented generation?",
    "How many parameters does BERT-large have?",
]

comparison_results = []

for q in test_questions:
    no_rag_answer = answer_without_rag(q)
    rag_result    = rag_answer(q)

    comparison_results.append({
        "question"      : q,
        "without_rag"   : no_rag_answer,
        "with_rag"      : rag_result["answer"],
        "source"        : list(set(rag_result["sources"]))
    })

    print(f"\n{'='*60}")
    print(f"❓ {q}")
    print(f"🚫 Without RAG: {no_rag_answer}")
    print(f"✅ With RAG   : {rag_result['answer']}")
    print(f"   Source: {comparison_results[-1]['source']}")


❓ How many attention heads does the Transformer use?
🚫 Without RAG: three
✅ With RAG   : 8
   Source: ['Attention Is All You Need']

❓ How many encoder layers does the base Transformer have?
🚫 Without RAG: three
✅ With RAG   : 6
   Source: ['Attention Is All You Need']

❓ What corpus was BERT pre-trained on?
🚫 Without RAG: san francisco
✅ With RAG   : unsupervised tasks
   Source: ['BERT: Pre-training of Deep Bidirectional Transformers']

❓ What is retrieval augmented generation?
🚫 Without RAG: Retrieval augmented generation
✅ With RAG   : a language model pre-training
   Source: ['Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks']

❓ How many parameters does BERT-large have?
🚫 Without RAG: three
✅ With RAG   : 340M
   Source: ['BERT: Pre-training of Deep Bidirectional Transformers', 'Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks']


We identified a retrieval limitation where the query 'What dataset was BERT pre-trained on?' failed to retrieve the chunk explicitly stating 'BooksCorpus and English Wikipedia,' despite this chunk being present in the indexed corpus. Diagnostic ranking revealed this chunk was ranked outside the top-k due to vocabulary mismatch: the query used 'dataset' while the source text used 'corpus.' This illustrates a known weakness of dense retrieval with lightweight embedding models, which can be mitigated through query reformulation, hybrid search (combining keyword and semantic search), or using larger embedding models such as all-mpnet-base-v2.

In [ ]:
import pandas as pd

df_results = pd.DataFrame(comparison_results)
df_results.to_csv("rag_comparison_results.csv", index=False)

print(df_results.to_string())

                                                  question                     without_rag                                          with_rag                                                                                                                     source
0       How many attention heads does the Transformer use?                           three                                                 8                                                                                                [Attention Is All You Need]
1  How many encoder layers does the base Transformer have?                           three                                                 6                                                                                                [Attention Is All You Need]
2                    What dataset was BERT pre-trained on?                            LSTM  unlabeled data over different pre-training tasks                                                                    

In [ ]:
trick_questions = [
    "What is the capital of Morocco?",          # not in the papers at all
    "Who won the World Cup in 2022?",            # completely unrelated topic
]

for q in trick_questions:
    result = rag_answer(q)
    print(f"❓ {q}")
    print(f"✅ With RAG: {result['answer']}")
    print(f"   Source: {set(result['sources'])}\n")

❓ What is the capital of Morocco?
✅ With RAG: Wikipedia
   Source: {'BERT: Pre-training of Deep Bidirectional Transformers', 'Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks'}

❓ Who won the World Cup in 2022?
✅ With RAG: Mexico
   Source: {'Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks'}



## Advanced Query Optimization (Query Rewriting Layer)

To maximize system robustness, a final Query Rewriting Layer is introduced into the pipeline.

Vector search is highly dependent on syntax; if an incoming query uses different vocabulary from the source documents, the FAISS lookup can yield subpar results. To overcome this, before running the similarity search, the raw user query is routed through an autonomous Llama-3 utility function designed to rewrite the prompt.

The model acts as an internal paraphrasing engine that replaces keywords with universal domain-specific synonyms and expands short phrases. This step ensures that the query vector aligns optimally with the document vectors, fetching high-quality contexts even when the user writes an ambiguous question.

In [ ]:
def build_better_prompt(query, context):
    return f"""You are a helpful assistant. Use ONLY the context below to answer the question in one clear sentence.

Context:
{context}

Question: {query}

Answer in one complete sentence:"""

results = search("What is retrieval augmented generation?", top_k=8)
context = "\n".join([r["text"] for r in results])
prompt = build_better_prompt("What is retrieval augmented generation?", context)
print(generate_answer(prompt, max_new_tokens=100))

RAG models which combine pre-trained parametric and non-parametric memory for language generation


In [ ]:
# find which chunk contains "BooksCorpus"
target_chunk_idx = None
for i, chunk in enumerate(all_chunks):
    if "BooksCorpus" in chunk["text"]:
        target_chunk_idx = i
        print(f"Found at chunk index: {i}")
        print(chunk["text"])
        break

# now check where FAISS ranks it for your query
query_vector = embedder.encode(["What dataset was BERT pre-trained on?"]).astype('float32')
distances, indices = index.search(query_vector, len(all_chunks))  # search ALL chunks

rank = list(indices[0]).index(target_chunk_idx)
print(f"\nThis chunk's actual rank: #{rank + 1} out of {len(all_chunks)}")
print(f"Its distance: {distances[0][rank]:.2f}")

Found at chunk index: 148
. Pre-training data The pre-training procedure largely follows the existing literature on language model pre-training. For the pre-training corpus we use the BooksCorpus (800M words) (Zhu et al., 2015) and English Wikipedia (2,500M words). For Wikipedia we extract only the text passages and ignore lists, tables, and headers

This chunk's actual rank: #58 out of 458
Its distance: 1.09


In [ ]:
from sentence_transformers import CrossEncoder

# this model scores (query, chunk) pairs directly — more accurate than embedding similarity
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def search_with_reranking(query, top_k_retrieve=10, top_k_final=3):
    # Step 1 — get more candidates than before with normal FAISS search
    candidates = search(query, top_k=top_k_retrieve)

    # Step 2 — re-score each candidate with the cross-encoder
    pairs = [[query, c["text"]] for c in candidates]
    scores = reranker.predict(pairs)

    # Step 3 — sort by the new, more accurate scores
    for c, s in zip(candidates, scores):
        c["rerank_score"] = float(s)

    reranked = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)

    return reranked[:top_k_final]

# test it on your problem case
results = search_with_reranking("What dataset was BERT pre-trained on?", top_k_retrieve=10, top_k_final=3)
for r in results:
    print(f"Score: {r['rerank_score']:.2f}")
    print(r['text'][:200])
    print()

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Score: 5.99
. Dur-ing pre-training, the model is trained on unlabeled data over different pre-training tasks. For fine-tuning, the BERT model is first initialized with the pre-trained parameters, and all of the p

Score: 4.50
. The most comparable existing pre-training method to BERT is OpenAI GPT, which trains a left-to-right Transformer LM on a large text cor-pus. In fact, many of the design decisions in BERT were intent

Score: 4.30
. As a re-sult, the pre-trained BERT model can be fine-tuned with just one additional output layer to create state-of-the-art models for a wide range of tasks, such as question answering and language 



In [ ]:
def rewrite_query(original_query):
    prompt = f"""Rewrite this question using different words but the same meaning,
to help search a technical paper. Original: "{original_query}"
Rewritten question:"""

    rewritten = generate_answer(prompt, max_new_tokens=50)
    return rewritten

# test it
original = "What dataset was BERT pre-trained on?"
rewritten = rewrite_query(original)
print(f"Original : {original}")
print(f"Rewritten: {rewritten}")

Original : What dataset was BERT pre-trained on?
Rewritten: What dataset was BERT pre-trained on?


In [ ]:
def advanced_rag_answer(query, top_k_retrieve=10, top_k_final=3):
    # Step 1 — rewrite the query
    rewritten = rewrite_query(query)

    # Step 2 — retrieve using BOTH original and rewritten query, merge results
    results_original  = search(query, top_k=top_k_retrieve)
    results_rewritten = search(rewritten, top_k=top_k_retrieve)

    # merge and deduplicate by chunk text
    seen = set()
    merged = []
    for r in results_original + results_rewritten:
        if r["text"] not in seen:
            merged.append(r)
            seen.add(r["text"])

    # Step 3 — re-rank the merged candidates
    pairs = [[query, c["text"]] for c in merged]
    scores = reranker.predict(pairs)
    for c, s in zip(merged, scores):
        c["rerank_score"] = float(s)

    reranked = sorted(merged, key=lambda x: x["rerank_score"], reverse=True)[:top_k_final]

    # Step 4 — generate the final answer
    context = "\n".join([c["text"] for c in reranked])
    prompt = f"""Answer the question based only on the context below.

Context: {context}

Question: {query}

Answer:"""

    answer = generate_answer(prompt, max_new_tokens=100)

    return {
        "query"           : query,
        "rewritten_query" : rewritten,
        "answer"          : answer,
        "sources"         : [c["source"] for c in reranked]
    }

# test on your problem case
result = advanced_rag_answer("What dataset was BERT pre-trained on?")
print(f"Original query  : {result['query']}")
print(f"Rewritten query  : {result['rewritten_query']}")
print(f"Answer           : {result['answer']}")
print(f"Sources          : {set(result['sources'])}")

Original query  : What dataset was BERT pre-trained on?
Rewritten query  : What dataset was BERT pre-trained on?
Answer           : unlabeled data
Sources          : {'BERT: Pre-training of Deep Bidirectional Transformers'}


In [ ]:
def rewrite_query(original_query):
    prompt = f"""Paraphrase the following question. Use a synonym for at least one key word.
Question: {original_query}
Paraphrase:"""

    rewritten = generate_answer(prompt, max_new_tokens=50)
    return rewritten

# test
original = "What dataset was BERT pre-trained on?"
rewritten = rewrite_query(original)
print(f"Original : {original}")
print(f"Rewritten: {rewritten}")

Original : What dataset was BERT pre-trained on?
Rewritten: What dataset was BERT pre-trained on?
